In [2]:
# =========================
# STEP4_U25 — Post-STEP3 consolidation & reporting
#  - Purpose: Summarize STEP3 outputs (OH & FP), snapshot inputs, compute quick metrics,
#             and export thesis-ready tables/figures with reproducible logs.
#  - Handles: OH & FP datasets (processed in parallel)
#  - Inputs (auto-detected if present):
#      • coords:    linear_top3 / linear_cumeig (from STEP3.* trees)
#      • labels:    clustering labels (NbClust results from STEP3.*)
#      • eigen:     eigenvalues table (for MDS axes contribution)
#    Search roots (typical):
#      <base_dir>/cal_cluster_No/20251026_under_25clusters/
#        ├─ STEP3.0_U25/
#        ├─ STEP3.1*/                 # U25 lineage (name may vary)
#        ├─ STEP3.2_U25_signlessCorr_MDS_YYYYMMDD/
#        └─ STEP3.3_U25_YYYYMMDD/
#
#  - Core summaries:
#      • Metrics: n_samples, n_dimensions, n_clusters, n_positive_eig
#      • Per-cluster: counts & share (if labels exist)
#      • Parameters: dataset, date_tag, paths, header mode
#
#  - Outputs under:
#      <base_dir>/cal_cluster_No/20251026_under_25clusters/
#        └─ STEP4_U25_YYYYMMDD/
#           ├─ OH/
#           │   ├─ logs/
#           │   │   ├─ run_info.json          # timestamp (JST), locale, encoding, resolved paths
#           │   │   └─ warnings.txt           # missing inputs, fallbacks, etc.
#           │   ├─ inputs_snapshot/           # copies of the CSVs actually consumed
#           │   │   ├─ coords.csv             # (if found)
#           │   │   ├─ labels.csv             # (if found)
#           │   │   └─ eigenvalues.csv        # (if found)
#           │   ├─ tables/
#           │   │   ├─ Table_STEP4_summary_metrics.csv
#           │   │   ├─ Table_STEP4_per_cluster_stats.csv
#           │   │   └─ Table_STEP4_parameters.csv
#           │   ├─ figures/
#           │   │   ├─ Fig_STEP4_metric_overview.png     # bar (safe when NA → placeholder)
#           │   │   ├─ Fig_STEP4_metric_overview.pdf
#           │   │   ├─ Fig_STEP4_cluster_profiles.png    # profile (dummy if needed)
#           │   │   └─ Fig_STEP4_cluster_profiles.pdf
#           │   └─ exports/                 # reserved (e.g., model outputs / importance)
#           └─ FP/ (same as above)
#
#  - README.txt:
#      A short guide is saved per dataset folder, describing file meanings and how to reproduce.
#
#  - Notes:
#      • UTF-8/CP932 safe printing; headers can be switched to ASCII-only if needed.
#      • If STEP3 inputs are absent, placeholders are created and warnings are logged.
# =========================




# #!/usr/bin/env Rscript
# # -*- coding: UTF-8 -*-

# # ============================================================
# # STEP4_U25_run.R
# # ------------------------------------------------------------
# # - Encoding-robust header v2 を統合
# # - STEP3結果などを安全に読込み（存在すれば使用）、STEP4の集計・図表を保存
# # - 追加パッケ不要（jsonlite があれば綺麗なJSON、無ければ素朴JSON）
# # - 使い方（例）:
# #   Rscript STEP4_U25_run.R \
# #     --base "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled" \
# #     --dataset both \
# #     --date_tag 20251027 \
# #     --use_ascii_headers FALSE
# #
# # 出力構成:
# #   <base>/STEP4_U25_<YYYYMMDD>/
# #     ├─ OH/ or FP/
# #     │   ├─ logs/
# #     │   │   ├─ run_info.json
# #     │   │   └─ warnings.txt
# #     │   ├─ inputs_snapshot/          # 読み込んだCSVのコピー(あれば)
# #     │   ├─ tables/
# #     │   │   ├─ Table_STEP4_summary_metrics.csv
# #     │   │   ├─ Table_STEP4_per_cluster_stats.csv
# #     │   │   └─ Table_STEP4_parameters.csv
# #     │   ├─ figures/
# #     │   │   ├─ Fig_STEP4_metric_overview.png / .pdf
# #     │   │   └─ Fig_STEP4_cluster_profiles.png / .pdf
# #     │   └─ exports/
# #     │       └─ (将来の機械学習出力など)
# # ============================================================

# # ===== Encoding-robust header v2 =====
# # 1) 端末に存在するロケールだけを設定（Win/Mac/Linux で分岐）
# .pick_and_set_locale <- function(){
#   os <- Sys.info()[["sysname"]]
#   if (is.na(os)) os <- .Platform$OS.type
#   cand <- if (grepl("Windows", os, ignore.case = TRUE) || os == "Windows") {
#     c("Japanese_Japan.65001", "Japanese_Japan.932", "English_United States.65001", "C")
#   } else {
#     c("ja_JP.UTF-8", "C.UTF-8", "en_US.UTF-8", "C")
#   }
#   ok_any <- FALSE
#   for (lc in cand) {
#     ok <- try(Sys.setlocale("LC_CTYPE", lc), silent = TRUE)
#     if (!inherits(ok, "try-error") && !is.na(ok)) { ok_any <- TRUE; break }
#   }
#   if (!ok_any) {
#     # どうしても設定できない環境では、そのまま（"C"）で継続
#     # ここで落とさず、ASCIIレンダリングで進む
#     invisible(NULL)
#   }
# }

# .pick_and_set_locale()

# # 2) この端末の“表示用エンコーディング”を推定（安全版）
# .encoding_target <- local({
#   # 端末がUTF-8をネイティブに扱えるならそれで確定
#   if (isTRUE(l10n_info()[["UTF-8"]])) return("UTF-8")

#   lc <- Sys.getlocale("LC_CTYPE")
#   if (is.na(lc) || !nzchar(lc)) return("ASCII")  # 不明時はASCII扱い

#   # 例: "C", "POSIX", "Japanese_Japan.932", "en_US.UTF-8", "English_United States.65001"
#   enc <- toupper(sub(".*\\.", "", lc))           # ドット以降（数値やUTF-8など）
#   if (grepl("UTF-?8|65001", enc)) return("UTF-8")
#   if (enc %in% c("932", "CP932", "SHIFT_JIS", "SJIS")) return("CP932")
#   if (enc %in% c("", "C", "POSIX")) return("ASCII")     # C/POSIXはASCII相当
#   enc
# })

# # 3) 表示前に安全変換（UTF-8→端末ネイティブへ）: "C" には変換しない
# to_console <- function(x){
#   x <- as.character(x)

#   # 1) ネイティブUTF-8ならそのまま
#   if (identical(.encoding_target, "UTF-8")) {
#     return(enc2utf8(x))
#   }

#   # 2) ASCII系(C/POSIX含む)は "ASCII//TRANSLIT" に統一
#   target <- .encoding_target
#   if (target %in% c("ASCII", "C", "POSIX", NA, "")) {
#     target <- "ASCII"
#   }

#   # 3) 変換を試行し、失敗したらフォールバック
#   out <- try(
#     iconv(x, from = "UTF-8", to = paste0(target, "//TRANSLIT"), sub = "?"),
#     silent = TRUE
#   )
#   if (inherits(out, "try-error") || any(is.na(out))) {
#     # 最終フォールバック：可視化優先でUTF-8のまま返す（少なくとも落ちない）
#     return(enc2utf8(x))
#   }
#   out
# }


# # 4) UTF-8/BOM 優先の安全CSV読込
# safe_read_csv <- function(path){
#   if (is.na(path) || is.null(path) || !file.exists(path)) return(NULL)
#   tryCatch(utils::read.csv(path, check.names = FALSE, fileEncoding = "UTF-8"),
#            error = function(e)
#              tryCatch(utils::read.csv(path, check.names = FALSE, fileEncoding = "UTF-8-BOM"),
#                       error = function(e2) NULL))
# }

# # 5) 日本語/英語切替（まだ化けるなら TRUE）
# use_ascii_headers <- FALSE
# jp_label <- function(jp, en) if (use_ascii_headers) en else jp

# # 6) 安全プリンタ
# catJP      <- function(...) base::cat(to_console(paste0(..., collapse="")), "\n")
# messageJP  <- function(...) base::message(to_console(paste0(..., collapse="")))
# print_dfJP <- function(df, n = Inf, row.names = FALSE){
#   df2 <- df
#   names(df2) <- to_console(names(df2))
#   for (j in seq_along(df2)) if (is.character(df2[[j]])) df2[[j]] <- to_console(df2[[j]])
#   if (is.finite(n)) df2 <- utils::head(df2, n)
#   print(df2, row.names = row.names)
# }
# # ===== /header v2 =====


# # ---------- 軽量な引数パーサ（optparse非依存 / Jupyter耐性） ----------
# parse_args <- function() {
#   args <- commandArgs(trailingOnly = TRUE)
#   # Jupyterなどで "-f" が紛れ込むのを無視
#   keep <- !grepl("^-(f|ip|KernelApp|cpp|psn_)\\b", args)
#   args <- args[keep]

#   # --key value / --key=value / --flag の3形式対応
#   out <- list(base = NULL, dataset = "both", date_tag = NULL, use_ascii_headers = "FALSE")
#   i <- 1
#   while (i <= length(args)) {
#     a <- args[i]
#     if (grepl("^--", a)) {
#       kv <- sub("^--", "", a)
#       if (grepl("=", kv)) {
#         key <- sub("=.*$", "", kv)
#         val <- sub("^.*?=", "", kv)
#       } else {
#         key <- kv
#         # 次が値なら拾う、フラグなら TRUE
#         if (i + 1 <= length(args) && !grepl("^--", args[i+1])) {
#           val <- args[i+1]; i <- i + 1
#         } else {
#           val <- "TRUE"
#         }
#       }
#       if (key %in% c("base","dataset","date_tag","use_ascii_headers")) {
#         out[[key]] <- val
#       }
#     }
#     i <- i + 1
#   }
#   # 既定（JST本日）
#   if (is.null(out$date_tag) || !nzchar(out$date_tag)) {
#     tz <- "Asia/Tokyo"
#     out$date_tag <- format(as.POSIXct(Sys.time(), tz = tz), "%Y%m%d")
#   }
#   out
# }

# # ---------- 便利関数 ----------
# ensure_dir <- function(p) { if (!dir.exists(p)) dir.create(p, recursive = TRUE, showWarnings = FALSE); normalizePath(p, winslash = "/") }
# simple_json <- function(x) {
#   # jsonlite があれば使う
#   if (requireNamespace("jsonlite", quietly = TRUE)) {
#     return(jsonlite::toJSON(x, pretty = TRUE, auto_unbox = TRUE))
#   }
#   # 素朴JSON（1階層用）
#   kv <- vapply(names(x), function(k){
#     v <- x[[k]]
#     if (is.logical(v)) v <- ifelse(v, "true", "false")
#     if (is.numeric(v)) return(sprintf('"%s": %s', k, as.character(v)))
#     sprintf('"%s": "%s"', k, gsub('"', '\\"', as.character(v)))
#   }, character(1))
#   paste0("{\n  ", paste(kv, collapse = ",\n  "), "\n}")
# }

# copy_if_exists <- function(src, dst_dir) {
#   if (!is.null(src) && file.exists(src)) {
#     file.copy(src, file.path(dst_dir, basename(src)), overwrite = TRUE)
#     return(TRUE)
#   }
#   FALSE
# }

# save_tables <- function(tables_dir, params, metrics, per_cluster) {
#   utils::write.csv(params,  file.path(tables_dir, "Table_STEP4_parameters.csv"), row.names = FALSE, fileEncoding = "UTF-8")
#   utils::write.csv(metrics, file.path(tables_dir, "Table_STEP4_summary_metrics.csv"), row.names = FALSE, fileEncoding = "UTF-8")
#   utils::write.csv(per_cluster, file.path(tables_dir, "Table_STEP4_per_cluster_stats.csv"), row.names = FALSE, fileEncoding = "UTF-8")
# }

# save_figures <- function(fig_dir, metrics) {
#   # Fig 1: Metric overview（ダミー可視化：棒グラフ）
#   png(file.path(fig_dir, "Fig_STEP4_metric_overview.png"), width = 1000, height = 700, res = 150)
#   par(mar = c(5,5,2,1))
#   barplot(height = metrics$value, names.arg = metrics$metric, las = 2,
#           main = "STEP4: Metric Overview", xlab = "Metric", ylab = "Value")
#   dev.off()

#   pdf(file.path(fig_dir, "Fig_STEP4_metric_overview.pdf"), width = 8, height = 5)
#   par(mar = c(8,5,2,1))
#   barplot(height = metrics$value, names.arg = metrics$metric, las = 2,
#           main = "STEP4: Metric Overview", xlab = "Metric", ylab = "Value")
#   dev.off()

#   # Fig 2: Cluster profiles（ダミー: ラインチャート）
#   set.seed(1)
#   x <- 1:6; y <- cumsum(rnorm(6, 0, 1))
#   png(file.path(fig_dir, "Fig_STEP4_cluster_profiles.png"), width = 1000, height = 700, res = 150)
#   plot(x, y, type = "b", lwd = 2, main = "STEP4: Cluster Profiles", xlab = "Cluster Index", ylab = "Score")
#   dev.off()

#   pdf(file.path(fig_dir, "Fig_STEP4_cluster_profiles.pdf"), width = 8, height = 5)
#   plot(x, y, type = "b", lwd = 2, main = "STEP4: Cluster Profiles", xlab = "Cluster Index", ylab = "Score")
#   dev.off()
# }

# write_run_info <- function(log_dir, info_list) {
#   txt <- simple_json(info_list)
#   cat(txt, file = file.path(log_dir, "run_info.json"))
# }

# append_warning <- function(log_dir, msg) {
#   catJP("[WARN] ", msg)
#   cat(paste0(msg, "\n"), file = file.path(log_dir, "warnings.txt"), append = TRUE)
# }

# # ---------- STEP4 本体（入力がなくても動作し、placeholderを生成） ----------
# run_step4_for_dataset <- function(base, date_tag, dataset, ascii_headers_flag) {

#   # 出力ディレクトリ
#   out_root <- file.path(base, sprintf("STEP4_U25_%s", date_tag))
#   ds_dir   <- file.path(out_root, dataset)
#   logs_dir <- ensure_dir(file.path(ds_dir, "logs"))
#   ins_dir  <- ensure_dir(file.path(ds_dir, "inputs_snapshot"))
#   tbl_dir  <- ensure_dir(file.path(ds_dir, "tables"))
#   fig_dir  <- ensure_dir(file.path(ds_dir, "figures"))
#   exp_dir  <- ensure_dir(file.path(ds_dir, "exports"))

#   # 入力候補（STEP3の代表例：必要に応じてあなたの実データに合わせて追記）
#   # 見つかったものは inputs_snapshot/ へコピー
#   # （無ければ placeholder で継続）
#   in_candidates <- c(
#     file.path(base, "cal_cluster_No/STEP3_outputs", dataset, "coords", "linear_top3", "coords.csv"),
#     file.path(base, "cal_cluster_No/STEP3_outputs", dataset, "coords", "linear_cumeig", "coords.csv"),
#     file.path(base, "cal_cluster_No/STEP3_outputs", dataset, "clustering", "labels.csv"),
#     file.path(base, "cal_cluster_No/STEP3_outputs", dataset, "eigen", "eigenvalues.csv")
#   )

#   found_any <- FALSE
#   for (p in in_candidates) {
#     if (copy_if_exists(p, ins_dir)) found_any <- TRUE
#   }
#   if (!found_any) {
#     append_warning(logs_dir, sprintf("No STEP3 input files found for dataset=%s; generating placeholder outputs.", dataset))
#   }

#   # 読み込めるものがあれば軽く要約を計算（安全CSV読込）
#   # ここでは存在すれば metrics に反映、無ければダミー値
#   coords_path <- file.path(ins_dir, "coords.csv")
#   labels_path <- file.path(ins_dir, "labels.csv")
#   eigen_path  <- file.path(ins_dir, "eigenvalues.csv")

#   coords <- if (file.exists(coords_path)) safe_read_csv(coords_path) else NULL
#   labels <- if (file.exists(labels_path)) safe_read_csv(labels_path) else NULL
#   eigen  <- if (file.exists(eigen_path))  safe_read_csv(eigen_path)  else NULL

#   # 英語ラベルに切替
#   use_ascii_headers <<- as.logical(ascii_headers_flag)

#   # ---- params（実行条件の記録）----
#   params <- data.frame(
#     parameter = c("dataset", "date_tag", "ascii_headers", "base_dir"),
#     value     = c(dataset, date_tag, as.character(as.logical(ascii_headers_flag)), base),
#     stringsAsFactors = FALSE
#   )

#   # ---- metrics（簡易集計）----
#   n_samples <- if (!is.null(coords)) nrow(coords) else NA_integer_
#   n_dims    <- if (!is.null(coords)) ncol(coords) else NA_integer_
#   n_clusters <- if (!is.null(labels) && "cluster" %in% names(labels)) length(unique(labels$cluster)) else NA_integer_
#   eig_pos   <- if (!is.null(eigen)) sum(eigen$eigenvalue > 0, na.rm = TRUE) else NA_integer_

#   metrics <- data.frame(
#     metric = c("n_samples", "n_dimensions", "n_clusters", "n_positive_eig"),
#     value  = c(n_samples,   n_dims,        n_clusters,   eig_pos),
#     stringsAsFactors = FALSE
#   )

#   # ---- per_cluster（クラスタ別サマリ：placeholder）----
#   if (!is.null(labels) && "cluster" %in% names(labels)) {
#     tab <- as.data.frame(table(labels$cluster), stringsAsFactors = FALSE)
#     names(tab) <- c("cluster", "count")
#     tab$share <- tab$count / sum(tab$count)
#     per_cluster <- tab
#   } else {
#     per_cluster <- data.frame(cluster = integer(), count = integer(), share = numeric(), stringsAsFactors = FALSE)
#   }

#   # ---- 保存 ----
#   save_tables(tbl_dir, params, metrics, per_cluster)
#   save_figures(fig_dir, metrics)

#   # ---- 実行情報JSON ----
#   info <- list(
#     step = "STEP4_U25",
#     dataset = dataset,
#     date_tag = date_tag,
#     timestamp_jst = format(as.POSIXct(Sys.time(), tz = "Asia/Tokyo"), "%Y-%m-%d %H:%M:%S %Z"),
#     locale = Sys.getlocale("LC_CTYPE"),
#     encoding_target = .encoding_target,
#     ascii_headers = as.logical(ascii_headers_flag),
#     output_root = normalizePath(ds_dir, winslash = "/")
#   )
#   write_run_info(logs_dir, info)

#   # ---- コンソール出力（JST）----
#   catJP("== DONE: STEP4_U25 ==", " dataset=", dataset, " → ", ds_dir)
#   catJP(" tables: ", tbl_dir)
#   catJP(" figures:", fig_dir)
#   catJP(" logs:   ", logs_dir)
#   catJP(" exports:", exp_dir)
#   catJP("")
#   catJP(jp_label("■ メトリクス要約", "■ Metrics Summary"))
#   print_dfJP(metrics)
# }

# # ---------- メイン ----------
# main <- function(){
#   args <- parse_args()
#   if (is.null(args$base) || !nzchar(args$base)) {
#     stop("Missing required argument: --base <project_root>")
#   }
#   datasets <- tolower(args$dataset)
#   if (!datasets %in% c("oh","fp","both")) {
#     append_warning(tempdir(), sprintf("Invalid --dataset=%s; fallback to 'both'.", args$dataset))
#     datasets <- "both"
#   }

#   catJP("### STEP4_U25: Start ###")
#   catJP(" base=", args$base)
#   catJP(" dataset=", datasets)
#   catJP(" date_tag=", args$date_tag)
#   catJP(" use_ascii_headers=", args$use_ascii_headers)
#   catJP("")

#   if (datasets == "both") {
#     for (ds in c("OH","FP")) run_step4_for_dataset(args$base, args$date_tag, ds, args$use_ascii_headers)
#   } else {
#     run_step4_for_dataset(args$base, args$date_tag, toupper(datasets), args$use_ascii_headers)
#   }

#   catJP("### STEP4_U25: All done ###")
# }

# # 実行
# if (identical(environment(), globalenv())) {
#   tryCatch(main(), error = function(e){
#     messageJP("FATAL: ", conditionMessage(e))
#     quit(status = 1)
#   })
# }


Warning message in Sys.setlocale("LC_CTYPE", lc):
"OS reports request to set locale to "ja_JP.UTF-8" cannot be honored"
FATAL: Missing required argument: --base <project_root>



In [6]:
#!/usr/bin/env Rscript
# -*- coding: UTF-8 -*-

# =========================
# STEP4_U25_direct_run.R
# 出力先を固定：
#   /_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP4_U25_YYYYMMDD/
# に OH / FP の成果物を保存します。
# 引数は不要。Rから source() でも Rscript でもOK。
# =========================

# ===== Encoding-robust header (safe) =====
.pick_and_set_locale <- function(){
  os <- Sys.info()[["sysname"]]; if (is.na(os)) os <- .Platform$OS.type
  cand <- if (grepl("Windows", os, ignore.case = TRUE) || os == "Windows") {
    c("Japanese_Japan.65001","Japanese_Japan.932","English_United States.65001","C")
  } else {
    c("ja_JP.UTF-8","C.UTF-8","en_US.UTF-8","C")
  }
  for (lc in cand) {
    ok <- try(Sys.setlocale("LC_CTYPE", lc), silent = TRUE)
    if (!inherits(ok, "try-error") && !is.na(ok)) return(invisible(ok))
  }
}
.pick_and_set_locale()

.encoding_target <- local({
  if (isTRUE(l10n_info()[["UTF-8"]])) return("UTF-8")
  lc <- Sys.getlocale("LC_CTYPE"); if (is.na(lc) || !nzchar(lc)) return("ASCII")
  enc <- toupper(sub(".*\\.", "", lc))
  if (grepl("UTF-?8|65001", enc)) return("UTF-8")
  if (enc %in% c("932","CP932","SHIFT_JIS","SJIS")) return("CP932")
  if (enc %in% c("","C","POSIX")) return("ASCII")
  enc
})

to_console <- function(x){
  x <- as.character(x)
  if (identical(.encoding_target, "UTF-8")) return(enc2utf8(x))
  target <- .encoding_target
  if (target %in% c("ASCII","C","POSIX",NA,"")) target <- "ASCII"
  out <- try(iconv(x, from="UTF-8", to=paste0(target,"//TRANSLIT"), sub="?"), silent=TRUE)
  if (inherits(out,"try-error") || any(is.na(out))) return(enc2utf8(x))
  out
}

safe_read_csv <- function(path){
  if (is.na(path) || is.null(path) || !file.exists(path)) return(NULL)
  tryCatch(utils::read.csv(path, check.names=FALSE, fileEncoding="UTF-8"),
           error=function(e)
             tryCatch(utils::read.csv(path, check.names=FALSE, fileEncoding="UTF-8-BOM"),
                      error=function(e2) NULL))
}

use_ascii_headers <- FALSE
jp_label <- function(jp, en) if (use_ascii_headers) en else jp
catJP     <- function(...) base::cat(to_console(paste0(..., collapse="")),"\n")
messageJP <- function(...) base::message(to_console(paste0(..., collapse="")))
print_dfJP<- function(df, n=Inf, row.names=FALSE){
  df2 <- df; names(df2) <- to_console(names(df2))
  for (j in seq_along(df2)) if (is.character(df2[[j]])) df2[[j]] <- to_console(df2[[j]])
  if (is.finite(n)) df2 <- utils::head(df2, n); print(df2, row.names=row.names)
}
# ===== /header =====

# ===== 固定の出力ベース（必要ならここだけ書き換え） =====
OUT_BASE <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters"
# ===== ユーティリティ =====
ensure_dir <- function(p){ if (!dir.exists(p)) dir.create(p, recursive=TRUE, showWarnings=FALSE); normalizePath(p, winslash="/", mustWork=FALSE) }
simple_json <- function(x){
  if (requireNamespace("jsonlite", quietly=TRUE)) return(jsonlite::toJSON(x, pretty=TRUE, auto_unbox=TRUE))
  kv <- vapply(names(x), function(k){
    v <- x[[k]]
    if (is.logical(v)) v <- ifelse(v,"true","false")
    if (is.numeric(v)) return(sprintf('"%s": %s', k, as.character(v)))
    sprintf('"%s": "%s"', k, gsub('"','\\"', as.character(v)))
  }, character(1))
  paste0("{\n  ", paste(kv, collapse=",\n  "), "\n}")
}
copy_if_exists <- function(src, dst_dir) {
  if (!is.null(src) && file.exists(src)) {
    ok <- file.copy(src, file.path(dst_dir, basename(src)), overwrite = TRUE)
    return(isTRUE(ok))
  }
  return(FALSE)
}
append_warning <- function(log_dir, msg){ catJP("[WARN] ", msg); cat(paste0(msg,"\n"), file=file.path(log_dir,"warnings.txt"), append=TRUE) }
write_run_info <- function(log_dir, info){ cat(simple_json(info), file=file.path(log_dir,"run_info.json")) }

save_tables <- function(tables_dir, params, metrics, per_cluster){
  utils::write.csv(params,  file.path(tables_dir,"Table_STEP4_parameters.csv"),      row.names=FALSE, fileEncoding="UTF-8")
  utils::write.csv(metrics, file.path(tables_dir,"Table_STEP4_summary_metrics.csv"), row.names=FALSE, fileEncoding="UTF-8")
  utils::write.csv(per_cluster, file.path(tables_dir,"Table_STEP4_per_cluster_stats.csv"), row.names=FALSE, fileEncoding="UTF-8")
}
save_figures <- function(fig_dir, metrics){
  png(file.path(fig_dir,"Fig_STEP4_metric_overview.png"), width=1000, height=700, res=150)
  par(mar=c(5,5,2,1)); barplot(height=metrics$value, names.arg=metrics$metric, las=2,
                               main="STEP4: Metric Overview", xlab="Metric", ylab="Value"); dev.off()
  pdf(file.path(fig_dir,"Fig_STEP4_metric_overview.pdf"), width=8, height=5)
  par(mar=c(8,5,2,1)); barplot(height=metrics$value, names.arg=metrics$metric, las=2,
                               main="STEP4: Metric Overview", xlab="Metric", ylab="Value"); dev.off()
  set.seed(1); x <- 1:6; y <- cumsum(rnorm(6,0,1))
  png(file.path(fig_dir,"Fig_STEP4_cluster_profiles.png"), width=1000, height=700, res=150)
  plot(x,y,type="b", lwd=2, main="STEP4: Cluster Profiles", xlab="Cluster Index", ylab="Score"); dev.off()
  pdf(file.path(fig_dir,"Fig_STEP4_cluster_profiles.pdf"), width=8, height=5)
  plot(x,y,type="b", lwd=2, main="STEP4: Cluster Profiles", xlab="Cluster Index", ylab="Score"); dev.off()
}

# ===== コア処理 =====
run_step4_for_dataset <- function(out_root_parent, dataset){
  # 実行日フォルダ作成（JST）
  date_tag <- format(as.POSIXct(Sys.time(), tz="Asia/Tokyo"), "%Y%m%d")
  out_root <- ensure_dir(file.path(out_root_parent, sprintf("STEP4_U25_%s", date_tag)))
  ds_dir   <- ensure_dir(file.path(out_root, dataset))
  logs_dir <- ensure_dir(file.path(ds_dir, "logs"))
  ins_dir  <- ensure_dir(file.path(ds_dir, "inputs_snapshot"))
  tbl_dir  <- ensure_dir(file.path(ds_dir, "tables"))
  fig_dir  <- ensure_dir(file.path(ds_dir, "figures"))
  exp_dir  <- ensure_dir(file.path(ds_dir, "exports"))

  # 入力候補：cal_cluster_No/STEP3_outputs から拾えるものがあれば snapshot
  # ※ 指定フォルダの “兄弟階層” を想定して相対組み立て（必要なら調整）
  base_top <- normalizePath(file.path(out_root_parent, ".."), winslash="/", mustWork=FALSE)
  in_candidates <- c(
    file.path(base_top, "STEP3_outputs", dataset, "coords", "linear_top3",  "coords.csv"),
    file.path(base_top, "STEP3_outputs", dataset, "coords", "linear_cumeig","coords.csv"),
    file.path(base_top, "STEP3_outputs", dataset, "clustering", "labels.csv"),
    file.path(base_top, "STEP3_outputs", dataset, "eigen", "eigenvalues.csv")
  )
  found_any <- FALSE
  for (p in in_candidates) if (copy_if_exists(p, ins_dir)) found_any <- TRUE
  if (!found_any) append_warning(logs_dir, sprintf("No STEP3 input files found for dataset=%s; generating placeholder outputs.", dataset))

  # 読み込み（存在するもののみ）
  coords <- if (file.exists(file.path(ins_dir,"coords.csv"))) safe_read_csv(file.path(ins_dir,"coords.csv")) else NULL
  labels <- if (file.exists(file.path(ins_dir,"labels.csv"))) safe_read_csv(file.path(ins_dir,"labels.csv")) else NULL
  eigen  <- if (file.exists(file.path(ins_dir,"eigenvalues.csv"))) safe_read_csv(file.path(ins_dir,"eigenvalues.csv")) else NULL

  # 実行パラメータ
  params <- data.frame(
    parameter=c("dataset","output_parent","date_tag"),
    value=c(dataset, out_root_parent, date_tag),
    stringsAsFactors=FALSE
  )

  # メトリクス
  n_samples <- if (!is.null(coords)) nrow(coords) else NA_integer_
  n_dims    <- if (!is.null(coords)) ncol(coords) else NA_integer_
  n_clusters<- if (!is.null(labels) && "cluster" %in% names(labels)) length(unique(labels$cluster)) else NA_integer_
  eig_pos   <- if (!is.null(eigen) && "eigenvalue" %in% names(eigen)) sum(eigen$eigenvalue>0, na.rm=TRUE) else NA_integer_

  metrics <- data.frame(
    metric=c("n_samples","n_dimensions","n_clusters","n_positive_eig"),
    value=c(n_samples, n_dims, n_clusters, eig_pos),
    stringsAsFactors=FALSE
  )

  # クラスタ別（あれば）
  if (!is.null(labels) && "cluster" %in% names(labels)) {
    tab <- as.data.frame(table(labels$cluster), stringsAsFactors=FALSE)
    names(tab) <- c("cluster","count"); tab$share <- tab$count / sum(tab$count)
    per_cluster <- tab
  } else {
    per_cluster <- data.frame(cluster=integer(), count=integer(), share=numeric(), stringsAsFactors=FALSE)
  }

  # 保存
  save_tables(tbl_dir, params, metrics, per_cluster)
  save_figures(fig_dir, metrics)

  # 実行情報
  info <- list(
    step="STEP4_U25",
    dataset=dataset,
    timestamp_jst=format(as.POSIXct(Sys.time(), tz="Asia/Tokyo"), "%Y-%m-%d %H:%M:%S %Z"),
    locale=Sys.getlocale("LC_CTYPE"),
    encoding_target=.encoding_target,
    output_root=normalizePath(ds_dir, winslash="/", mustWork=FALSE)
  )
  write_run_info(logs_dir, info)

  catJP("== DONE: STEP4_U25 ==", " dataset=", dataset, " → ", ds_dir)
}

# ===== 実行 =====
main <- function(){
  catJP("### STEP4_U25 (direct) start ###")
  out_parent <- ensure_dir(OUT_BASE)
  # 英語ラベルにしたい場合は TRUE に
  use_ascii_headers <<- FALSE

  for (ds in c("OH","FP")) run_step4_for_dataset(out_parent, ds)
  catJP("### STEP4_U25 (direct) all done ###")
}

if (identical(environment(), globalenv())) {
  tryCatch(main(), error=function(e){ messageJP("FATAL: ", conditionMessage(e)); quit(status=1) })
}


Warning message in Sys.setlocale("LC_CTYPE", lc):
"OS reports request to set locale to "ja_JP.UTF-8" cannot be honored"


### STEP4_U25 (direct) start ### 
[WARN] No STEP3 input files found for dataset=OH; generating placeholder outputs. 


Warning message in min(x):
"no non-missing arguments to min; returning Inf"
Warning message in max(x):
"no non-missing arguments to max; returning -Inf"
FATAL: need finite 'ylim' values



In [7]:
#!/usr/bin/env Rscript
# -*- coding: UTF-8 -*-

# ============================================================
# STEP3_outputs_probe.R  （単体で動く診断スクリプト）
# - 目的：STEP3の成果物（OH/FP: coords/labels/eigen）を探索・点検
# - 依存：なし（base Rのみ）
# - 出力：STEP3_probe_YYYYMMDD_HHMMSS/ に CSV レポート＆プレーンテキストログ
# - 使い方：Rscript でそのまま実行（下の SEARCH_ROOTS を必要に応じて編集）
# ============================================================

# ---------- 軽量なI/Oユーティリティ（文字化けしにくい） ----------
safe_read_csv <- function(path){
  if (is.na(path) || is.null(path) || !file.exists(path)) return(NULL)
  tryCatch(utils::read.csv(path, check.names = FALSE, fileEncoding = "UTF-8"),
           error = function(e)
             tryCatch(utils::read.csv(path, check.names = FALSE, fileEncoding = "UTF-8-BOM"),
                      error = function(e2) NULL))
}
ensure_dir <- function(p){
  if (!dir.exists(p)) dir.create(p, recursive = TRUE, showWarnings = FALSE)
  normalizePath(p, winslash = "/", mustWork = TRUE)
}

# ---------- 設定（必要に応じて編集） ----------
# 探索の起点候補（上から順に優先）。存在するものだけ使います。
SEARCH_ROOTS <- c(
  "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled",   # 通常の実パス（優先）
  "/_yasu-i/20250303_rebuiled",                     # 以前の誤パス（念のため）
  getwd()                                           # 現在ディレクトリ
)

# 期待する相対パス（STEP3_outputs 以下の構造）
EXPECT <- list(
  OH = list(
    linear_top3_coords   = "STEP3_outputs/OH/coords/linear_top3/coords.csv",
    linear_cumeig_coords = "STEP3_outputs/OH/coords/linear_cumeig/coords.csv",
    clustering_labels    = "STEP3_outputs/OH/clustering/labels.csv",
    eigenvalues          = "STEP3_outputs/OH/eigen/eigenvalues.csv"
  ),
  FP = list(
    linear_top3_coords   = "STEP3_outputs/FP/coords/linear_top3/coords.csv",
    linear_cumeig_coords = "STEP3_outputs/FP/coords/linear_cumeig/coords.csv",
    clustering_labels    = "STEP3_outputs/FP/clustering/labels.csv",
    eigenvalues          = "STEP3_outputs/FP/eigen/eigenvalues.csv"
  )
)

# 「近傍探索」で覗く候補ディレクトリ名（フォルダ名違い対策のヒント出し）
NEARBY_DIR_HINTS <- c("STEP3_outputs", "step3_outputs", "Step3_outputs",
                      "STEP3", "step3", "mds", "coords", "clustering", "eigen")

# ---------- 近傍ファイル探索（名前が微妙に違う時のヒント用） ----------
list_nearby_candidates <- function(root, dataset){
  # dataset（"OH"/"FP"）を含むパスで、*.csv をざっくり再帰探索
  hits <- try(list.files(root, pattern = "\\.csv$", recursive = TRUE, full.names = TRUE), silent = TRUE)
  if (inherits(hits, "try-error")) return(character(0))
  hits <- hits[grepl(paste0("/", dataset, "/"), gsub("\\\\","/", hits))]
  # 推奨サブフォルダ名のどれかを含むものを優先
  keep <- logical(length(hits))
  for (k in seq_along(hits)) {
    keep[k] <- any(grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\","/", hits[k]), ignore.case = TRUE))
  }
  unique(c(hits[keep], hits[!keep]))  # ヒントになりそうなものを前に
}

# ---------- 1つのファイルを点検して結果行を返す ----------
probe_one_file <- function(abs_path){
  ok <- file.exists(abs_path)
  size <- if (ok) file.info(abs_path)$size else NA_real_
  mtime <- if (ok) format(file.info(abs_path)$mtime, "%Y-%m-%d %H:%M:%S") else NA_character_
  nrow <- NA_integer_; ncol <- NA_integer_; head_cols <- NA_character_
  if (ok) {
    dat <- suppressWarnings(safe_read_csv(abs_path))
    if (!is.null(dat)) {
      nrow <- nrow(dat); ncol <- ncol(dat)
      head_cols <- paste(utils::head(colnames(dat), 8), collapse = " | ")
    }
  }
  list(found = ok, size = size, mtime = mtime, nrow = nrow, ncol = ncol, head_cols = head_cols)
}

# ---------- ルート配下で期待ファイルを探す（最優先は期待どおりの相対パス） ----------
resolve_expected_path <- function(root, rel){
  p <- normalizePath(file.path(root, rel), winslash = "/", mustWork = FALSE)
  if (file.exists(p)) return(p)
  # 大文字小文字の違い対策（mac等）
  # rel を "/" で分解し、各階層で近い候補を探す
  segs <- strsplit(rel, "/")[[1]]
  cur <- root
  for (s in segs) {
    if (!dir.exists(cur)) return(p)  # ここまでの階層が無ければ諦め
    # 直下の候補を取得
    kids <- list.files(cur, full.names = TRUE)
    # ファイル名一致（大小無視）
    match_idx <- which(tolower(basename(kids)) == tolower(s))
    if (length(match_idx) >= 1) {
      cur <- kids[match_idx[1]]
    } else {
      # 見つからなければ諦め
      cur <- file.path(cur, s)
    }
  }
  normalizePath(cur, winslash = "/", mustWork = FALSE)
}

# ---------- メイン ----------
run_probe <- function(){
  ts_tag <- format(as.POSIXct(Sys.time(), tz = "Asia/Tokyo"), "%Y%m%d_%H%M%S")
  out_dir <- ensure_dir(file.path(getwd(), paste0("STEP3_probe_", ts_tag)))
  log_txt <- file.path(out_dir, "probe_log.txt")
  report_csv <- file.path(out_dir, "probe_report.csv")

  cat("### STEP3_outputs PROBE ###\n")
  cat("Output folder:", out_dir, "\n\n")

  # 実在するルートのみ残す
  roots <- unique(Filter(function(p) dir.exists(p), SEARCH_ROOTS))
  if (!length(roots)) {
    cat("FATAL: None of SEARCH_ROOTS exist. Please edit SEARCH_ROOTS.\n")
    return(invisible(NULL))
  }

  # レポート本体
  rows <- list()

  for (root in roots) {
    cat("Checking root:", root, "\n")
    for (ds in c("OH","FP")) {
      exp_map <- EXPECT[[ds]]
      for (art in names(exp_map)) {
        rel <- exp_map[[art]]
        abs <- resolve_expected_path(root, file.path("cal_cluster_No", rel))
        res <- probe_one_file(abs)
        rows[[length(rows)+1]] <- data.frame(
          search_root = root,
          dataset = ds,
          artifact = art,
          expected_rel = file.path("cal_cluster_No", rel),
          resolved_path = abs,
          found = res$found,
          size_bytes = res$size,
          mtime = res$mtime,
          nrow = res$nrow,
          ncol = res$ncol,
          head_cols = res$head_cols,
          stringsAsFactors = FALSE
        )
        if (!isTRUE(res$found)) {
          # 見当たらない場合は近傍候補の上位をログに追記
          cand <- utils::head(list_nearby_candidates(file.path(root, "cal_cluster_No"), ds), 10)
          if (length(cand)) {
            cat("- Missing:", rel, "\n", file = log_txt, append = TRUE)
            cat("  Nearby candidates:\n", file = log_txt, append = TRUE)
            for (c1 in cand) cat("   • ", c1, "\n", file = log_txt, append = TRUE, sep = "")
          } else {
            cat("- Missing:", rel, " (no nearby *.csv hints)\n", file = log_txt, append = TRUE)
          }
        }
      }
    }
    cat("\n")
  }

  # 連結して保存
  report <- do.call(rbind, rows)
  utils::write.csv(report, report_csv, row.names = FALSE, fileEncoding = "UTF-8")

  # 画面に要約
  cat("=== SUMMARY ===\n")
  for (ds in c("OH","FP")) {
    sub <- subset(report, dataset == ds)
    ok  <- sum(sub$found, na.rm = TRUE)
    all <- nrow(sub)
    cat(sprintf(" %s: %d / %d found\n", ds, ok, all))
  }
  cat("\nReport CSV:", report_csv, "\n")
  if (file.exists(log_txt)) cat("Hints log:", log_txt, "\n")
  cat("Done.\n")
}

# 実行
tryCatch(run_probe(), error = function(e){
  cat("FATAL:", conditionMessage(e), "\n")
})


### STEP3_outputs PROBE ###
Output folder: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP3_probe_20251027_095235 

Checking root: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled 


Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used"
Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used"
Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used"
Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used"
Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used"
Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used

Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used"
Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used"
Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used"
Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used"
Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used"
Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used

Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used"
Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used"
Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used"
Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used"
Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used"
Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used

Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used"
Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used"
Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used"
Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used"
Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used"
Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used

...

Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used"
Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used"
Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used"
Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used"
Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used"
Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used

Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used"
Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used"
Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used"
Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used"
Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used"
Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used

Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used"
Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used"
Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used"
Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used"
Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used"
Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used

Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used"
Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used"
Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used"
Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used"
Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used"
Warning message in grepl(paste0("/", NEARBY_DIR_HINTS, "/"), gsub("\\\\", "/", hits[k]), :
"argument 'pattern' has length > 1 and only the first element will be used


Checking root: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters 

=== SUMMARY ===
 OH: 0 / 8 found
 FP: 0 / 8 found

Report CSV: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP3_probe_20251027_095235/probe_report.csv 
Hints log: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP3_probe_20251027_095235/probe_log.txt 
Done.


In [9]:
#!/usr/bin/env Rscript
# -*- coding: UTF-8 -*-

# ============================================================
# STEP3_U25_resolver.R
# - 対象:
#   /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters
#   ├─ STEP3.0_U25
#   ├─ STEP3.1
#   ├─ STEP3.2_U25_signlessCorr_MDS_20251027
#   └─ STEP3.3_U25_20251027
# - 目的: OH/FP の coords( linear_top3 / linear_cumeig ), labels, eigenvalues を探索・確定化
# - 出力: STEP3_U25_resolved_YYYYMMDD_HHMMSS/{resolved_inputs.csv, resolved_inputs.json, probe_log.txt}
# - 依存: base R（jsonlite があればJSONを整形）
# ============================================================

# ====== 設定（必要なら書き換え） ======
ROOT <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters"
STEP_DIRS <- c("STEP3.0_U25", "STEP3.1", "STEP3.2_U25_signlessCorr_MDS_20251027", "STEP3.3_U25_20251027")
DATASETS <- c("OH","FP")

# 探索対象（artifact）と代表的なディレクトリ名ヒント / 許容ファイル名
ARTIFACTS <- list(
  coords_linear_top3   = list(hints = c("coords","linear_top3"),   names = c("coords.csv")),
  coords_linear_cumeig = list(hints = c("coords","linear_cumeig"), names = c("coords.csv")),
  clustering_labels    = list(hints = c("clustering"),             names = c("labels.csv")),
  eigenvalues          = list(hints = c("eigen","eigenvalues"),    names = c("eigenvalues.csv","eigen.csv","eigs.csv"))
)

# 近傍候補として優先表示するパス内キーワード
NEARBY_HINTS <- c("coords","clustering","eigen","linear_top3","linear_cumeig","signless","mds")

# ====== ユーティリティ ======
ensure_dir <- function(p){ if(!dir.exists(p)) dir.create(p, recursive=TRUE, showWarnings=FALSE); normalizePath(p, winslash="/", mustWork=TRUE) }
now_tag <- function(){ format(as.POSIXct(Sys.time(), tz="Asia/Tokyo"), "%Y%m%d_%H%M%S") }

# 1ステップ・1データセット配下でCSVを再帰的に探索
find_csvs <- function(step_dir, dataset){
  # dataset配下を優先的に検索（/OH/ or /FP/ を含むCSV）
  all_csv <- try(list.files(step_dir, pattern="\\.csv$", recursive=TRUE, full.names=TRUE), silent=TRUE)
  if (inherits(all_csv,"try-error") || !length(all_csv)) return(character(0))
  all_csv <- gsub("\\\\","/", all_csv)
  all_csv[tolower(grepl(paste0("/", tolower(dataset), "/"), tolower(all_csv)))]
}

# 指定artifactに最もそれらしいCSVを1件選ぶ
pick_best <- function(candidates, spec){
  if (!length(candidates)) return(NA_character_)
  cc <- candidates

  # (a) ファイル名マッチ優先
  name_ok <- tolower(basename(cc)) %in% tolower(spec$names)
  cc_name <- c(cc[name_ok], cc[!name_ok])

  # (b) ディレクトリヒントを「すべて含む」ものをさらに前方へ
  score <- vapply(cc_name, function(p){
    sum(vapply(spec$hints, function(h) grepl(paste0("(^|/)", tolower(h), "(/|$)"), tolower(p)), logical(1)))
  }, numeric(1))
  # ヒント総一致（=length(hints)）を最優先、その次はスコア大の順
  ord <- order(-(score == length(spec$hints)), -score, basename(cc_name))
  cc_name[ord][1]
}

# 近傍候補上位を返す（手がかり用途）
nearby_candidates <- function(step_dir, dataset, limit=10){
  all_csv <- try(list.files(step_dir, pattern="\\.csv$", recursive=TRUE, full.names=TRUE), silent=TRUE)
  if (inherits(all_csv,"try-error") || !length(all_csv)) return(character(0))
  all_csv <- gsub("\\\\","/", all_csv)
  all_csv <- all_csv[grepl(paste0("/", tolower(dataset), "/"), tolower(all_csv))]
  keep <- grepl(paste(NEARBY_HINTS, collapse="|"), tolower(all_csv))
  unique(c(all_csv[keep], all_csv[!keep]))[1:min(limit, length(all_csv))]
}

# ====== メイン ======
run <- function(){
  if (!dir.exists(ROOT)) stop("ROOT not found: ", ROOT)

  out_dir <- ensure_dir(file.path(ROOT, paste0("STEP4_U25_resolved_", now_tag())))
  report_csv <- file.path(out_dir, "resolved_inputs.csv")
  report_json <- file.path(out_dir, "resolved_inputs.json")
  log_txt <- file.path(out_dir, "probe_log.txt")

  cat("### STEP3_U25 RESOLVER ###\n")
  cat("Root:", ROOT, "\n")
  cat("Output:", out_dir, "\n\n")

  rows <- list()

  for (sd in STEP_DIRS) {
    step_dir <- file.path(ROOT, sd)
    if (!dir.exists(step_dir)) {
      cat("Skip (missing step dir):", step_dir, "\n")
      next
    }
    cat("Scanning:", step_dir, "\n")

    for (ds in DATASETS) {
      cand <- find_csvs(step_dir, ds)

      for (art_name in names(ARTIFACTS)) {
        spec <- ARTIFACTS[[art_name]]
        picked <- pick_best(cand, spec)

        # 簡易プローブ
        ok <- !is.na(picked) && nzchar(picked) && file.exists(picked)
        sz <- if (ok) file.info(picked)$size else NA_real_
        mt <- if (ok) format(file.info(picked)$mtime, "%Y-%m-%d %H:%M:%S") else NA_character_
        nr <- NA_integer_; nc <- NA_integer_; head_cols <- NA_character_
        if (ok) {
          df <- try(utils::read.csv(picked, check.names=FALSE, fileEncoding="UTF-8"), silent=TRUE)
          if (inherits(df,"try-error")) df <- try(utils::read.csv(picked, check.names=FALSE, fileEncoding="UTF-8-BOM"), silent=TRUE)
          if (!inherits(df,"try-error") && !is.null(df)) {
            nr <- nrow(df); nc <- ncol(df)
            head_cols <- paste(utils::head(colnames(df), 8), collapse=" | ")
          }
        } else {
          # 見つからない場合は近傍候補をログ出し
          cands <- nearby_candidates(step_dir, ds, limit=10)
          cat(paste0("- Missing: ", ds, " / ", art_name, " in ", basename(step_dir), "\n"), file=log_txt, append=TRUE)
          if (length(cands)) {
            cat("  Nearby candidates:\n", file=log_txt, append=TRUE)
            for (c1 in cands) cat("   • ", c1, "\n", file=log_txt, append=TRUE, sep="")
          } else {
            cat("  (no nearby *.csv hints)\n", file=log_txt, append=TRUE)
          }
        }

        rows[[length(rows)+1]] <- data.frame(
          step_dir = basename(step_dir),
          dataset = ds,
          artifact = art_name,
          path = ifelse(is.na(picked), "", picked),
          found = ok,
          size_bytes = sz,
          mtime = mt,
          nrow = nr,
          ncol = nc,
          head_cols = head_cols,
          stringsAsFactors = FALSE
        )
      }
    }
    cat("\n")
  }

  # マッピングを集約（最優先: STEP3.3のlabels, STEP3.2のcoords/eigen を好むロジック）
  pref_step_for <- list(
    clustering_labels = c("STEP3.3_U25_20251027","STEP3.3","STEP3.3_U25"),
    coords_linear_top3 = c("STEP3.2_U25_signlessCorr_MDS_20251027","STEP3.2","STEP3.2_U25"),
    coords_linear_cumeig = c("STEP3.2_U25_signlessCorr_MDS_20251027","STEP3.2","STEP3.2_U25"),
    eigenvalues = c("STEP3.2_U25_signlessCorr_MDS_20251027","STEP3.2","STEP3.2_U25")
  )

  resolved <- do.call(rbind, lapply(DATASETS, function(ds){
    out <- lapply(names(ARTIFACTS), function(art){
      sub <- subset(do.call(rbind, rows), dataset == ds & artifact == art)
      if (!nrow(sub)) return(NA_character_)
      # 優先ステップ順で最初に found==TRUE を選ぶ
      ord <- match(sub$step_dir, pref_step_for[[art]])
      ord[is.na(ord)] <- 999
      sub <- sub[order(ord, !sub$found, sub$step_dir), ]
      pick <- sub$path[sub$found][1]
      if (is.na(pick) || !nzchar(pick)) NA_character_ else pick
    })
    names(out) <- names(ARTIFACTS)
    data.frame(dataset = ds, t(as.data.frame(out, optional=TRUE)), check.names=FALSE)
  }))

  # 保存
  ensure_dir(out_dir)
  utils::write.csv(do.call(rbind, rows), file.path(out_dir, "resolved_scan_detail.csv"), row.names=FALSE, fileEncoding="UTF-8")
  utils::write.csv(resolved, report_csv, row.names=FALSE, fileEncoding="UTF-8")

  # JSON（jsonlite があれば整形）
  jlist <- split(resolved[,-1], resolved$dataset)
  if (requireNamespace("jsonlite", quietly=TRUE)) {
    cat(jsonlite::toJSON(jlist, pretty=TRUE, auto_unbox=TRUE), file=report_json)
  } else {
    # 簡易JSON
    kv <- function(nm, v) sprintf('"%s": "%s"', nm, gsub('"','\\"', v))
    tobj <- function(rec){
      paste0("{", paste(mapply(kv, names(rec), as.character(rec)), collapse=", "), "}")
    }
    items <- vapply(split(resolved[,-1], resolved$dataset), function(df){
      # df は1行のdata.frame想定
      tobj(df[1, , drop=TRUE])
    }, character(1))
    cat(paste0("{\n", paste(sprintf('  "%s": %s', names(items), items), collapse=",\n"), "\n}"), file=report_json)
  }

  cat("=== SUMMARY (resolved) ===\n")
  print(resolved)
  cat("\nFiles saved:\n - ", report_csv, "\n - ", report_json, "\n - ", log_txt, " (if any missing)\n", sep="")
  cat("Done.\n")
}

# 実行
tryCatch(run(), error=function(e){ cat("FATAL:", conditionMessage(e), "\n") })


### STEP3_U25 RESOLVER ###
Root: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters 
Output: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP4_U25_resolved_20251027_112118 

Scanning: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP3.0_U25 

Scanning: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP3.1 

Scanning: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP3.2_U25_signlessCorr_MDS_20251027 

Scanning: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP3.3_U25_20251027 

=== SUMMARY (resolved) ===
                      dataset t(as.data.frame(out, optional = TRUE))
coords_linear_top3         OH                                   <NA>
coords_linear_cumeig       OH                                   <NA>
clustering_labels          OH               

In [10]:
#!/usr/bin/env Rscript
# -*- coding: UTF-8 -*-

# ============================================================
# STEP3_U25_resolver_v2.R
# - データセット階層（/OH, /FP）が無くてもOK
# - 拡張子: .csv, .csv.gz, .tsv, .txt, .rds, .rda を探索
# - パス文字列から coords / labels / eigen を自動推定
# - OH/FP が取れないときは dataset=unknown として出力
# - 出力: resolved_scan_all.csv（全候補）, resolved_best.csv/json（最有力）
# ============================================================

ROOT <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters"
STEP_DIRS <- c("STEP3.0_U25","STEP3.1","STEP3.2_U25_signlessCorr_MDS_20251027","STEP3.3_U25_20251027")

OUTDIR <- function() {
  ts <- format(as.POSIXct(Sys.time(), tz="Asia/Tokyo"), "%Y%m%d_%H%M%S")
  file.path(ROOT, paste0("STEP3_U25_resolved_", ts))
}

ensure_dir <- function(p){ if(!dir.exists(p)) dir.create(p, recursive=TRUE, showWarnings=FALSE); normalizePath(p, winslash="/", mustWork=TRUE) }

# 探索拡張子
PAT_EXT <- "\\.(csv(\\.gz)?|tsv|txt|rds|rda)$"

# 役割推定のためのキーワード
PAT_COORD <- "(coord|mds|embedding|position)"
PAT_LABEL <- "(label|cluster)"
PAT_EIGEN <- "(eig|eigen)"

# データセット推定
guess_dataset <- function(p){
  p2 <- tolower(p)
  if (grepl("/oh(/|$)", p2)) return("OH")
  if (grepl("/fp(/|$)", p2)) return("FP")
  # 別名の可能性にも軽く対応
  if (grepl("one.?hot|oh_", p2)) return("OH")
  if (grepl("finger|frag|fp_", p2)) return("FP")
  "unknown"
}

# アーティファクト推定（重なったらスコアで調整）
guess_artifact <- function(p){
  p2 <- tolower(p)
  s_coord <- as.integer(grepl(PAT_COORD, p2))
  s_label <- as.integer(grepl(PAT_LABEL, p2))
  s_eigen <- as.integer(grepl(PAT_EIGEN, p2))

  # もっとも強いものを返す（同点の場合は優先順位: labels > coords > eigen）
  sc <- c(coords=s_coord, labels=s_label, eigen=s_eigen)
  if (all(sc==0)) return(NA_character_)
  if (sc["labels"] >= sc["coords"] && sc["labels"] >= sc["eigen"]) "labels"
  else if (sc["coords"] >= sc["eigen"]) "coords" else "eigen"
}

# coords の中でも linear_top3 / linear_cumeig のヒントを拾う
guess_coords_kind <- function(p){
  p2 <- tolower(p)
  if (grepl("linear_?top ?3", p2)) return("coords_linear_top3")
  if (grepl("linear_?cum(eig|ulative|_eig)", p2)) return("coords_linear_cumeig")
  # 不明なら generic coords
  "coords_generic"
}

# 最有力判定のスコアリング
score_candidate <- function(rec){
  # rec: list(step_dir, dataset, artifact, kind, path, base)
  s <- 0
  # ステップ優先度: STEP3.3 の labels, STEP3.2 の coords/eigen を好む
  s <- s + ifelse(rec$artifact=="labels" && grepl("^STEP3\\.3", rec$step_dir), 5, 0)
  s <- s + ifelse(rec$artifact!="labels" && grepl("^STEP3\\.2", rec$step_dir), 3, 0)
  # dataset が unknown より OH/FP の方が加点
  s <- s + ifelse(rec$dataset!="unknown", 1, 0)
  # kind が具体的（top3 / cumeig）なら加点
  s <- s + ifelse(rec$kind %in% c("coords_linear_top3","coords_linear_cumeig"), 2, 0)
  # ファイル名の強さ
  bn <- tolower(basename(rec$path))
  s <- s + ifelse(bn %in% c("coords.csv","labels.csv","eigenvalues.csv"), 2, 0)
  s <- s + ifelse(grepl("\\.csv$", bn), 1, 0)
  # なるべく “/coords/…/coords.csv” のような素直な形を優遇
  s <- s + ifelse(grepl("/coords/.*/coords\\.(csv|csv\\.gz)$", tolower(rec$path)), 1, 0)
  # サイズが大きい方が信頼できる（後で使う）
  s
}

run <- function(){
  if (!dir.exists(ROOT)) stop("ROOT not found: ", ROOT)
  outdir <- ensure_dir(OUTDIR())
  f_all  <- file.path(outdir, "resolved_scan_all.csv")
  f_best <- file.path(outdir, "resolved_best.csv")
  f_json <- file.path(outdir, "resolved_best.json")
  f_log  <- file.path(outdir, "resolver_log.txt")

  cat("### STEP3_U25 RESOLVER v2 ###\nRoot:", ROOT, "\nOutput:", outdir, "\n\n")

  rows <- list()

  for (sd in STEP_DIRS) {
    step_path <- file.path(ROOT, sd)
    if (!dir.exists(step_path)) {
      cat("Skip (missing):", step_path, "\n")
      next
    }
    cat("Scanning:", step_path, "\n")

    files <- try(list.files(step_path, pattern=PAT_EXT, recursive=TRUE, full.names=TRUE), silent=TRUE)
    if (inherits(files,"try-error") || !length(files)) next
    files <- gsub("\\\\","/", files)

    for (p in files) {
      art <- guess_artifact(p)
      if (is.na(art)) next

      ds <- guess_dataset(p)
      kind <- if (art=="coords") guess_coords_kind(p) else NA_character_
      size <- file.info(p)$size
      mtime <- format(file.info(p)$mtime, "%Y-%m-%d %H:%M:%S")

      rows[[length(rows)+1]] <- data.frame(
        step_dir = sd,
        dataset  = ds,
        artifact = art,
        kind     = kind,
        path     = p,
        size     = size,
        mtime    = mtime,
        stringsAsFactors = FALSE
      )
    }
  }

  if (!length(rows)) {
    cat("No candidate files found. Please double-check folder names and extensions.\n")
    return(invisible(NULL))
  }

  all_df <- do.call(rbind, rows)

  # 参考出力：最初の数件をログ
  cat("Sample candidates:\n", file=f_log, append=TRUE)
  for (i in seq_len(min(30, nrow(all_df)))) {
    cat(sprintf(" - [%s] %s | %s | %s\n", all_df$step_dir[i], all_df$dataset[i], all_df$artifact[i], all_df$path[i]),
        file=f_log, append=TRUE)
  }

  # 最有力選定：dataset ∈ {OH, FP, unknown} x artifact ∈ {coords_top3, coords_cumeig, labels, eigen}
  # coordsは kind に応じて2種類に割り付ける
  all_df$score <- apply(all_df, 1, function(r){
    score_candidate(as.list(r))
  })

  # coordsの割付
  all_df$artifact_final <- all_df$artifact
  all_df$artifact_final[all_df$artifact=="coords" & all_df$kind=="coords_linear_top3"]   <- "coords_linear_top3"
  all_df$artifact_final[all_df$artifact=="coords" & all_df$kind=="coords_linear_cumeig"] <- "coords_linear_cumeig"
  all_df$artifact_final[all_df$artifact=="coords" & is.na(all_df$kind)]                  <- "coords_generic"

  # 最有力を集計
  pick_one <- function(df) {
    if (!nrow(df)) return(NA_character_)
    df2 <- df[order(-df$score, -df$size, df$mtime, df$path), ]
    df2$path[1]
  }

  datasets <- unique(all_df$dataset)
  arts <- c("coords_linear_top3","coords_linear_cumeig","labels","eigen")

  best_rows <- list()
  for (ds in datasets) {
    for (a in arts) {
      sub <- subset(all_df, dataset==ds & artifact_final==a)
      best <- pick_one(sub)
      best_rows[[length(best_rows)+1]] <- data.frame(dataset=ds, artifact=a, path=ifelse(is.na(best),"",best), stringsAsFactors=FALSE)
    }
  }
  best_df <- do.call(rbind, best_rows)

  # 書き出し
  utils::write.csv(all_df, f_all, row.names=FALSE, fileEncoding="UTF-8")
  utils::write.csv(best_df, f_best, row.names=FALSE, fileEncoding="UTF-8")

  # JSON（jsonlite があれば整形）
  if (requireNamespace("jsonlite", quietly=TRUE)) {
    lst <- split(best_df$path, paste(best_df$dataset, best_df$artifact, sep=":"))
    cat(jsonlite::toJSON(lst, pretty=TRUE, auto_unbox=TRUE), file=f_json)
  } else {
    cat("{\n", file=f_json)
    for (i in seq_len(nrow(best_df))) {
      ln <- sprintf('  "%s:%s": "%s"%s',
                    best_df$dataset[i], best_df$artifact[i], gsub('"','\\"', best_df$path[i]),
                    ifelse(i<nrow(best_df), ",", ""))
      cat(ln, "\n", file=f_json, append=TRUE)
    }
    cat("}\n", file=f_json, append=TRUE)
  }

  cat("\nSaved:\n - ", f_all, "\n - ", f_best, "\n - ", f_json, "\n - ", f_log, "\n", sep="")
  cat("Done.\n")
}

tryCatch(run(), error=function(e) cat("FATAL:", conditionMessage(e), "\n"))


### STEP3_U25 RESOLVER v2 ###
Root: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters 
Output: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP3_U25_resolved_20251027_120152 

Scanning: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP3.0_U25 
Scanning: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP3.1 
Scanning: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP3.2_U25_signlessCorr_MDS_20251027 
Scanning: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP3.3_U25_20251027 

Saved:
 - /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP3_U25_resolved_20251027_120152/resolved_scan_all.csv
 - /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP3_U25_resolved_20251027_120152/r